In [40]:
import torch

print(torch.version.cuda)

12.6


In [39]:
import torch
from torch.utils.data import DataLoader, Dataset
from torch import nn
from torch.profiler import profile, ProfilerActivity, record_function

# ----- 간단 모델 -----
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.seq = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )
    def forward(self, x):
        return self.seq(self.flatten(x))

class MyDataset(Dataset):
    def __init__(self):
        self.x = torch.randn(64, 1, 28, 28)
        self.y = torch.randint(0, 10, (64,))
    
    def __getitem__(self, index):
        return self.x[index], self.y[index]

    def __len__(self):
        return len(self.x)


# ----- 준비 -----
device = "cuda" if torch.cuda.is_available() else "cpu"
model  = NeuralNetwork().to(device)
opt    = torch.optim.SGD(model.parameters(), lr=1e-2)
crit   = nn.CrossEntropyLoss()



# x = torch.randn(64, 1, 28, 28, device=device)
# y = torch.randint(0, 10, (64,), device=device)

# ----- 프로파일링: 거시적 구간 라벨링 -----
with profile(
    activities=[ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if torch.cuda.is_available() else []),
    with_stack=True,
    record_shapes=True,
    profile_memory=True,
) as prof:
    with record_function("00.data"):    # 데이터 준비/전처리
        my_data = MyDataset()
        data_wrapped_by_DataLoader = DataLoader(my_data, batch_size=64)
        xb, yb = next(iter(data_wrapped_by_DataLoader))
        xb = xb.to('cuda')
        yb = yb.to('cuda')

    with record_function("10.forward(nn.Module)"):
        logits = model(xb)

    with record_function("20.loss(nn.* or F.*)"):
        loss = crit(logits, yb)

    with record_function("30.backward(autograd)"):
        loss.backward()

    with record_function("40.optimizer.step()"):
        opt.step()

    with record_function("50.optimizer.zero_grad()"):
        opt.zero_grad(set_to_none=True)

print(prof.key_averages().table(sort_by="self_cpu_time_total", row_limit=20))
prof.export_chrome_trace("trace.json")
print("✅ trace.json 저장 완료 → Chrome에서 chrome://tracing → [Load] 로 열기.")


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                  30.backward(autograd)        16.89%       1.327ms        17.55%       1.379ms       1.379ms           0 B           0 B       2.30 MB       2.30 MB             1  
                                        cudaMemcpyAsync         9.51%     747.400us         9.51%     747.400us     373.700us           0 B           0 B           0 B           0 B             2  
         

In [ ]:
# torch의 call_graph가 나오지 않아서 실패
# torch.profiler을 사용하여 재도전.

from pycallgraph2 import PyCallGraph
from pycallgraph2.output import GraphvizOutput
from torch import nn

class NeuralNetwork(nn.Module):
    def __init__(self, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU() ,
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )
    
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

with PyCallGraph(output=GraphvizOutput(output_file='torch_callgraph.png')):
    NeuralNetwork()

In [ ]:
# main함수만 나와서 실패. torch라이브러리를 인식하지 못하는 문제인듯

# callgraph_torch.py
# pip install pycallgraph2 graphviz
# (mac) brew install graphviz  /  (ubuntu) sudo apt-get install graphviz

from pycallgraph2 import PyCallGraph, Config
from pycallgraph2.output import GraphvizOutput
from pycallgraph2.globbing_filter import GlobbingFilter

import torch
from torch import nn
import torch.optim as optim

class NeuralNetwork(nn.Module):
    def __init__(self, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )
    
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

def run_once():
    # 모델/데이터
    model = NeuralNetwork()
    x = torch.randn(32, 1, 28, 28)           # 32개 배치
    y = torch.randint(0, 10, (32,))          # 라벨

    # 손실/옵티마이저
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=1e-2)

    # 1 step 학습 루프 (forward -> loss -> backward -> step)
    out = model(x)
    loss = criterion(out, y)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

if __name__ == "__main__":
    graphviz = GraphvizOutput(output_file="torch_callgraph.png")
    config = Config()
    config.trace_filter = GlobbingFilter(
        include=[
            "__main__.*",     # 내 코드
            "torch.*",        # torch 전체 (nn, optim 등)
        ],
        exclude=[
            "torch._C.*",     # C++ 바인딩(파이썬 밖이라 콜그래프엔 노이즈)
            "site.*",
            "importlib.*",
            "pkg_resources.*",
            "threading.*",
            "concurrent.*",
            "multiprocessing.*",
        ],
    )

    with PyCallGraph(output=graphviz, config=config):
        run_once()
